In [0]:
query = """
    SELECT
      ROW_NUMBER() OVER (ORDER BY ci.customer_id) AS customer_number,
      ci.customer_id,
      ci.customer_key,
      ci.first_name,
      ci.last_name,
      ci.marital_status,
      ci.gender,
      ca.birth_date as birthdate,
      la.country,
      ci.create_date
    FROM silver.crm_customers ci
    LEFT JOIN silver.erp_customer_location la
        ON ci.customer_key = la.customer_number
    LEFT JOIN silver.erp_customers ca
        ON ci.customer_key = ca.customer_number  
"""
df = spark.sql(query)

In [0]:
df.limit(10).display()

In [0]:
%sql
SELECT
  customer_id,
  COUNT(*)
FROM gold.dim_customers
GROUP BY customer_id
HAVING COUNT(*)>1 
LIMIT 30

In [0]:
df = spark.read.table("workspace.gold.dim_customers")

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col

In [0]:
query = """
    SELECT *
    FROM
      (select *,
            ROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY create_date DESC) as rnk
        FROM gold.dim_customers)
    WHERE rnk = 1
      """
df = spark.sql(query)
df.display()

In [0]:
total_count = df.count()
distinct_count = df.select("customer_id").distinct().count()
if total_count > distinct_count:
  print(f"Ai duplicate! Diferența este de {total_count - distinct_count} rânduri.")
else:
  print("There are no duplicate records in the table.")

# Writing Gold Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.gold.dim_customers")

# Sanity Check

In [0]:
%sql
SELECT *
FROM gold.dim_customers LIMIT 30;